# **Shapley interaction indices: SII, k-SII, STII, FSII and the shapiq package**

Practice for the module [“Feature interactions”](https://ai-interpretability.school/learn/8/0/0), lesson “SHAP values for assessing interaction”, step “Concrete indices: from SII to higher orders”.

In the lesson, interaction indices were weights in a common framework: SII is not efficient on groups, k-SII, STII and FSII each fix this in their own way, and on the same game they give different numbers. Here we compute all of them with the [shapiq](https://github.com/mmschlk/shapiq) package ([Muschalik et al., 2024](https://arxiv.org/abs/2410.01649)):

1. a three-player game with known Möbius dividends: four indices and an efficiency check;
2. a bike rental demand model and pairwise SHAP interaction values from `shap` next to SII from shapiq;
3. four indices on one object and different coalition value functions;
4. third-order interactions;
5. approximate computation: the error of shapiq approximators versus the budget;
6. the global picture: the mean absolute k-SII over a sample;
7. where it breaks: correlated features.

The notebook runs on CPU; the whole run takes two to three minutes.

In [ ]:
!pip install -q shapiq==1.7.0 shap==0.52.0

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm without ipywidgets; Colab has ipywidgets
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import shapiq
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some CPUs
SEED = 0
pd.set_option('display.width', 140)

## 1. A game with known dividends

The Möbius dividends (Harsanyi dividends) $m(S)$ are what a coalition $S$ brings beyond all its subcoalitions; every game decomposes into them: $v(T)=\sum_{S\subseteq T}m(S)$. We define a three-player game directly by its dividends:

$$m(\{0\})=1,\quad m(\{1\})=1,\quad m(\{0,1\})=2,\quad m(\{0,1,2\})=3,$$

all other dividends are zero. Player 2 brings nothing alone and takes part only in the triple interaction. The total payoff is $v(N)-v(\varnothing)=1+1+2+3=7$.

Second-order indices have to split the triple dividend 3 somehow between pairs and singletons, and they split it differently. `shapiq.ExactComputer` computes any index by exhaustive enumeration of coalitions.

In [ ]:
def toy_game(coalitions):
    # coalitions: boolean matrix (number of coalitions, 3); value is the sum of the dividends of included subsets
    c = coalitions.astype(float)
    return c[:, 0] + c[:, 1] + 2 * c[:, 0] * c[:, 1] + 3 * c[:, 0] * c[:, 1] * c[:, 2]


exact_toy = shapiq.ExactComputer(toy_game, n_players=3)
toy = {idx: exact_toy(index=idx, order=2) for idx in ('SII', 'k-SII', 'STII', 'FSII')}
subsets = [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2)]
toy_tab = pd.DataFrame({idx: [iv[s] for s in subsets] for idx, iv in toy.items()},
                       index=[str(s) for s in subsets])
print(toy_tab.round(3))
print('Möbius dividends:', {k: round(v, 3) for k, v in exact_toy(index='Moebius', order=3).dict_values.items() if abs(v) > 1e-9})

**Task 1. For each of the four indices compute the sum of all order-1 and order-2 values (the six numbers in the table column) and compare with the total payoff 7. Which index is not efficient? Then compute k-SII of third order (`exact_toy(index='k-SII', order=3)`) and compare its values with the Möbius dividends.**

In [ ]:
# Your code here

How the four indices split the triple dividend: STII gives the lower orders exactly their dividends and spreads the whole triple dividend evenly over the three pairs; k-SII and FSII give each pair half of the triple dividend, and the singletons get a correction that makes the decomposition efficient; SII gives the pairs the same as k-SII, but its singletons are ordinary Shapley values, and the sum does not add up.

## 2. A bike demand model

[Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) (Fanaee-T and Gama, 2013): 17,379 hours of the Capital Bikeshare rental in Washington in 2011–2012, the outcome is the number of rides per hour. The file is in `data/` of the course repository: it is the copy from the shapiq repository, and the original UCI data are distributed under the CC BY 4.0 license. Features: hour of day `hour`, whether it is a working day `workingday`, temperature `temp` and felt temperature `feel_temp` (°C), humidity, wind, weather, season, month, weekday, holiday, year. We predict the logarithm of the ride count with a random forest.

The interaction here is known in advance: on working days demand has two peaks, morning and evening, and on days off one, in the afternoon. So the effect of the hour depends on whether it is a working day.

In [ ]:
bike = pd.read_csv('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/bike.csv')
for c in ('season', 'weather'):
    bike[c] = bike[c].astype('category').cat.codes
y = np.log1p(bike.pop('count').values)
names = list(bike.columns)
X = bike.values.astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED)
rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
print('features:', names)
print(f'R² on the held-out part: {rf.score(Xte, yte):.3f}')
H, W, TEMP, FEEL = (names.index(n) for n in ('hour', 'workingday', 'temp', 'feel_temp'))

hours = np.arange(24)
for wd, color in ((1, '#3b6ea5'), (0, '#e67e22')):
    m = X[:, W] == wd
    plt.plot(hours, [np.expm1(y[m & (X[:, H] == h)]).mean() for h in hours], 'o-', color=color,
             label='working day' if wd else 'day off')
plt.xlabel('Hour of day')
plt.ylabel('Mean rides per hour')
plt.title('Bike sharing: demand by hour')
plt.legend()
plt.show()

## Pairwise SHAP interaction values and SII

The lesson says: the pairwise $\phi_{ij}$ from `shap` is a symmetrized SII, $\phi_{ij}=\tfrac12\mathrm{SII}_{ij}$. We check this on one hour of the held-out part: `shap.TreeExplainer(rf).shap_interaction_values` and `shapiq.TreeExplainer(..., index='SII', max_order=2)` are both exact tree algorithms with the same coalition value function (path-dependent: missing features are “switched off” by the shares of training objects in the tree branches).

We take a working day, 8 a.m., the morning peak.

In [ ]:
i0 = np.flatnonzero((Xte[:, W] == 1) & (Xte[:, H] == 8))[0]
x0 = Xte[i0]
print('chosen hour:', {n: float(v) for n, v in zip(names, x0)})
print(f'model prediction: {rf.predict(x0[None])[0]:.3f} (log of the ride count)')

phi_pairs = shap.TreeExplainer(rf).shap_interaction_values(x0[None])[0]       # M×M matrix
sii_tree = shapiq.TreeExplainer(model=rf, index='SII', max_order=2).explain(x0)

**Task 2. For all 66 feature pairs $(i, j)$, $i<j$, compute the largest absolute discrepancy between `2 * phi_pairs[i, j]` and `sii_tree[(i, j)]`. Check the second property from the lesson “SHAP values for assessing interaction” as well: the sum of row $i$ of `phi_pairs` equals the ordinary Shapley value of feature $i$ (take `shap.TreeExplainer(rf).shap_values`). Print the pair with the largest absolute SII and its value.**

In [ ]:
# Your code here

## 3. Four indices on one object and the coalition value function

The shapiq tree algorithm does not cover all indices. To compare all four we build the game explicitly: the value of a coalition $S$ is the mean forest prediction when the features in $S$ are taken from our hour and the rest from 100 random hours of the training part (`shapiq.MarginalImputer`, an interventional value function). Twelve features means $2^{12}=4096$ coalitions, and `ExactComputer` enumerates all of them in a few seconds.

This is a **different** value function than the path-dependent one in the tree algorithm, so SII of section 2 and SII here are different numbers of the same index.

In [ ]:
background = Xtr[np.random.default_rng(SEED).choice(len(Xtr), 100, replace=False)]
game = shapiq.MarginalImputer(rf.predict, data=background, x=x0, sample_size=100, random_state=SEED)
exact = shapiq.ExactComputer(game, n_players=len(names))
ivs = {idx: exact(index=idx, order=2) for idx in ('SII', 'k-SII', 'STII', 'FSII')}
full = rf.predict(x0[None])[0] - rf.predict(background).mean()
print(f'total payoff v(N) − v(∅) = prediction − mean prediction on background: {full:.4f}')
for idx, iv in ivs.items():
    print(f'{idx:>6}: sum of orders 1 and 2 = {sum(v for k, v in iv.dict_values.items() if len(k) > 0):.4f}')

top_pairs = sorted(pairs, key=lambda p: -abs(ivs['k-SII'][p]))[:6]
cmp = pd.DataFrame({idx: [iv[p] for p in top_pairs] for idx, iv in ivs.items()},
                   index=[f'{names[i]} × {names[j]}' for i, j in top_pairs])
cmp['SII, path-dependent (section 2)'] = [sii_tree[p] for p in top_pairs]
print(cmp.round(4))

The pairs nearly coincide across the four indices: third-order interactions are weaker than pairwise ones in this model (section 4), and there are no triple dividends to redistribute. The singleton values differ more: for SII they are ordinary Shapley values, for the others effects with a share of the interactions subtracted. The figure below shows the singleton values of the four indices for the six features with the largest absolute k-SII.

In [ ]:
top_single = sorted(range(len(names)), key=lambda i: -abs(ivs['k-SII'][(i,)]))[:6]
width = 0.2
fig, ax = plt.subplots(figsize=(10, 3.8))
for k, (idx, color) in enumerate(zip(ivs, ['#7f8c8d', '#3b6ea5', '#e67e22', '#27ae60'])):
    ax.bar(np.arange(6) + (k - 1.5) * width, [ivs[idx][(i,)] for i in top_single], width, color=color, label=idx)
ax.set_xticks(range(6))
ax.set_xticklabels([names[i] for i in top_single])
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Order-1 value')
ax.set_title('Working day, 8 a.m.: singleton values of four order-2 indices')
ax.legend()
plt.show()

## 4. Third-order interactions

`shapiq.TreeExplainer` with `index='k-SII'` and `max_order=3` decomposes the prediction up to triples. For k-SII the sum of all values of orders 1–3 again equals the prediction minus the baseline value.

In [ ]:
k3_tree = shapiq.TreeExplainer(model=rf, index='k-SII', max_order=3).explain(x0)
by_order = {o: sum(v for k, v in k3_tree.dict_values.items() if len(k) == o) for o in (1, 2, 3)}
print('sum of values by order:', {o: round(v, 4) for o, v in by_order.items()})
print(f'prediction − baseline value: {rf.predict(x0[None])[0] - k3_tree.baseline_value:.4f}')
triples = sorted([k for k in k3_tree.dict_values if len(k) == 3], key=lambda k: -abs(k3_tree[k]))[:5]
for k in triples:
    print(f'{" × ".join(names[i] for i in k):>40}: {k3_tree[k]:+.4f}')

## 5. Approximate computation: error versus budget

Enumerating $2^M$ coalitions is feasible with twelve features but not with fifty. shapiq approximators estimate an index from `budget` evaluations of the game. We compare four of them with the exact order-2 k-SII of section 3 on the same game: `KernelSHAPIQ` (weighted regression over coalitions), `SVARMIQ` (stratified sampling), `SHAPIQ` (unbiased coalition sampling) and `PermutationSamplingSII` (permutations). The error is the mean squared deviation over the 66 pairs.

**Task 3. For budgets 128, 256, 512, 1024, 2048 and 4096 and each approximator, compute the mean squared error over the pairs relative to `ivs['k-SII']` (seed `random_state=SEED`, `max_order=2`, `index='k-SII'`; `PermutationSamplingSII` also takes `index='k-SII'`). Plot the error against the budget on log axes. Which approximator is the most accurate at budget 256?**

In [ ]:
# Your code here

At a small budget (128 and 256 game evaluations) permutation sampling is the most accurate, from 512 on KernelSHAPIQ is. At budget 4096 every approximator except the permutation one sees all coalitions and returns the exact value: the error drops to machine zero. Permutation sampling cannot enumerate all coalitions and converges more slowly than the others as the budget grows.

## 6. The global picture

Indices are about one object. A summary over the data is the mean of absolute values: the mean $|\text{k-SII}|$ of every pair over 300 hours of the held-out part, computed with the exact tree algorithm.

In [ ]:
tree_k2 = shapiq.TreeExplainer(model=rf, index='k-SII', max_order=2)
sample = Xte[np.random.default_rng(SEED).choice(len(Xte), 300, replace=False)]
mean_abs = pd.Series(0.0, index=[f'{names[i]} × {names[j]}' for i, j in pairs])
for xs in sample:
    iv = tree_k2.explain(xs)
    mean_abs += np.array([abs(iv[p]) for p in pairs]) / len(sample)
top = mean_abs.sort_values().tail(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.index, top.values, color='#3b6ea5')
ax.set_xlabel('Mean |k-SII| over 300 held-out hours')
ax.set_title('Strongest pairwise interactions of the bike demand model')
plt.tight_layout()
plt.show()
print(top[::-1].round(4))

## 7. Where it breaks: correlated features

`temp` and `feel_temp` nearly duplicate each other. The model can rely on either, and the “hour × temperature” interaction is split between two pairs, “hour × temp” and “hour × feel_temp”, the way the forest decided rather than the way the world works. We check this: train the same forest without `feel_temp` and compare the mean $|\text{k-SII}|$ of the pairs with hour.

**Task 4. Compute the correlation of `temp` and `feel_temp` on the training part. Train a forest with the same hyperparameters without the `feel_temp` column and compute the mean $|\text{k-SII}|$ of the pair `hour × temp` over the same 300 hours `sample`. Compare with the sum of the mean $|\text{k-SII}|$ of the pairs `hour × temp` and `hour × feel_temp` of the full model.**

In [ ]:
# Your code here

**Task 5. From the outputs of sections 2, 3 and 7: name three choices that have to be stated next to an interaction index value so that it can be read.**

**Your answer:**

## What to take away

- **The indices differ in how they distribute higher-order interactions.** On a game with a triple dividend, SII, k-SII, STII and FSII give different singleton and pairwise values, and SII does not add up to the total payoff.
- **$\phi_{ij}$ from `shap` is half of SII.** Both exact tree algorithms give the same result up to machine zero, and the rows of the `shap` matrix add up to Shapley values.
- **The coalition value function changes the numbers more than the choice of index.** Path-dependent and interventional games give different values of the same index on the same object.
- **Approximate computation needs a budget, and the best approximator depends on it.** On twelve features, the errors of shapiq approximators at the same budget differ by two orders of magnitude; at a small budget permutation sampling is more accurate, at a large one KernelSHAPIQ.
- **Correlated features share an interaction.** The number of a pair depends on what other features the model saw.

Sources:
1. [Muschalik et al., 2024. shapiq: Shapley interactions for machine learning](https://arxiv.org/abs/2410.01649)
2. [Bordt and von Luxburg, 2023. From Shapley values to generalized additive models and back](https://proceedings.mlr.press/v206/bordt23a.html)
3. [Sundararajan et al., 2020. The Shapley Taylor interaction index](https://arxiv.org/abs/1902.05622)
4. [Tsai, Yeh, and Ravikumar, 2023. Faith-Shap: the faithful Shapley interaction index](https://arxiv.org/abs/2203.00870)
5. [Fanaee-T and Gama, 2013. Event labeling combining ensemble detectors and background knowledge](https://doi.org/10.1007/s13748-013-0040-3), the Bike Sharing data